# 00 — Introducción y visión del proyecto

**Audiencia:** tutor de tesis / personas sin conocimiento previo de packing.

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

| Grupo benchmark | Instancia | Diferenciación esperada |
|-----------------|-----------|-------------------------|
| 3D-BPP | `showcase_3d_bpp_instance.json` | best_fit ~27 emp. vs extreme_points ~22 |
| Híbrido (misma instancia 3D) | `showcase_3d_bpp_instance.json` | compaction/LS mueven 8–11 piezas |
| Mejora 3D (misma instancia 3D) | `showcase_3d_bpp_instance.json` | relocation/swap/orientation/bin_reduction refinan layout |
| Container Loading | `showcase_container_loading_instance.json` | weight_aware ~27 vs single_container ~17 |
| Single Container | `showcase_single_container_instance.json` | best_fit ~13 (94% util) vs first_fit ~16 |
| Cartonization | `showcase_cartonization_instance.json` | single-box vs multi_box |
| Palletization | `showcase_palletization_instance.json` | layer_based vs stack_based |
| Stacking-aware | `showcase_stacking_aware_instance.json` | stacking_aware vs stack_based |

## ¿Qué problema resolvemos?

Dado un **catálogo de piezas** (cajas con dimensiones y cantidades) y uno o más **contenedores**,
hay que decidir **dónde colocar cada pieza** (posición y rotación) para:
- Maximizar el aprovechamiento del espacio
- Respetar peso máximo y no solapar piezas
- Reportar qué piezas no cupieron

Eso pertenece a la familia *Cutting and Packing*, con **cuatro formulaciones** en este proyecto:
3D-BPP, Container Loading, Cartonization y Single Container Loading.

## La brecha que identifiqué

Existen muchos algoritmos, pero cada implementación usa **formatos distintos**.
No se pueden comparar de forma justa ni integrar en un espacio de datos.

## Mi propuesta

Plataforma de **servicios** donde cada algoritmo implementado expone su propio endpoint ``POST /api/v1/algorithms/{algorithm_name}/execute`` con entrada/salida JSON homogénea.
El **benchmark** usa perfiles estándar (`constructive`, `hybrid`, `box_selection`, …) para comparar ≥2 motores sobre la misma instancia.


In [ ]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


In [ ]:
import subprocess, sys
from _shared.loaders import load_showcase_instance, SHOWCASE_INSTANCES, BENCHMARK_GROUPS
from packing_services import SERVICE_VERSION
from packing_services.algorithms.registry import get_default_registry
from packing_services.domain.enums import AlgorithmStatus

display.show_problem_overview(load_showcase_instance('3D_BPP'))

registry = get_default_registry()
n_impl = len(registry.list_metadata(status=AlgorithmStatus.IMPLEMENTED))
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', str(ROOT / 'tests'), '--collect-only', '-q',
     '--ignore=tests/test_api.py'],
    capture_output=True, text=True, cwd=ROOT,
)
n_tests = next((int(l.split()[0]) for l in test_run.stdout.splitlines() if 'tests collected' in l), '?')
print(f'\n=== packing-services v{SERVICE_VERSION} ===')
n_algo_svc = len([s for s in runners.run_dataspace_catalog().services if s.service_name.endswith('Algorithm Service')])
print(f'Algoritmos implementados: {n_impl} | Servicios por algoritmo: {n_algo_svc} | Tests: {n_tests}')
print(f'Tipos de problema: {len(SHOWCASE_INSTANCES)} | Grupos de benchmark: {len(BENCHMARK_GROUPS)}')
from _shared.loaders import SHOWCASE_DIFFERENTIATION
import pandas as pd
from IPython.display import display as ipy_display
rows = [{'Grupo': g, 'Diferenciación esperada': SHOWCASE_DIFFERENTIATION.get(g, '—')} for g in BENCHMARK_GROUPS]
ipy_display(pd.DataFrame(rows).style.hide(axis='index'))


In [ ]:
print('═══ Vista previa: benchmarks homogéneos (mismas instancias en todos los notebooks) ═══')
for group in BENCHMARK_GROUPS:
    display.run_and_show_showcase_benchmark(group, show_chart=False)


## Puente: informe técnico → código

| Decisión del informe | Evidencia en el repositorio |
|----------------------|----------------------------|
| Priorizar heurísticas constructivas | 4 heurísticas 3D-BPP + py3dbp opcional |
| Mejora local reutilizable | `solution_compaction`, `constructive_plus_local_search` |
| Validador geométrico propio | `PackingValidator` + tests |
| Servicios independientes | 5 agregados + 1 endpoint por algoritmo |
| Preparación espacio de datos | `GET /api/v1/services` + `GET /api/v1/algorithms/{name}` |

**Siguiente:** `01_arquitectura_y_servicios.ipynb`
